In [0]:
BRONZE_TABLE_NAME = "workspace.marketing_campaign.bronze_marketing_campaign"
SILVER_TABLE_NAME = "workspace.marketing_campaign.silver_marketing_campaign"

bronze_df = spark.table(BRONZE_TABLE_NAME)

print(f"Bronze table: {BRONZE_TABLE_NAME}")
print(f"Silver table: {SILVER_TABLE_NAME}")
print(f"Bronze rows: {bronze_df.count()}")

In [0]:
def to_snake_case(column_name):
    return (
        column_name
        .strip()
        .replace(" ", "_")
        .replace("-", "_")
        .lower()
    )

silver_df = bronze_df

for old_column_name in bronze_df.columns:
    new_column_name = to_snake_case(old_column_name)
    silver_df = silver_df.withColumnRenamed(old_column_name, new_column_name)

print(silver_df.columns)

In [0]:
from pyspark.sql.functions import col, to_date

silver_df = (
    silver_df
    .withColumn("dt_customer", to_date(col("dt_customer"), "dd-MM-yyyy"))
)

In [0]:
from pyspark.sql.functions import median

income_median = silver_df.selectExpr("percentile_approx(income, 0.5)").collect()[0][0]

silver_df = silver_df.fillna({"income": income_median})

print(f"Income median used for missing values: {income_median}")

In [0]:
from pyspark.sql.functions import current_date, year, datediff, when

silver_df = (
    silver_df
    .withColumn("customer_age", year(current_date()) - col("year_birth"))
    .withColumn("customer_tenure_days", datediff(current_date(), col("dt_customer")))
    .withColumn(
        "has_children",
        when((col("kidhome") + col("teenhome")) > 0, 1).otherwise(0)
    )
    .withColumn(
        "total_spend",
        col("mntwines")
        + col("mntfruits")
        + col("mntmeatproducts")
        + col("mntfishproducts")
        + col("mntsweetproducts")
        + col("mntgoldprods")
    )
    .withColumn(
        "total_purchases",
        col("numwebpurchases")
        + col("numcatalogpurchases")
        + col("numstorepurchases")
    )
    .withColumn(
        "accepted_previous_campaign",
        when(
            (col("acceptedcmp1") + col("acceptedcmp2") + col("acceptedcmp3") + col("acceptedcmp4") + col("acceptedcmp5")) > 0,
            1
        ).otherwise(0)
    )
)

In [0]:
silver_df = silver_df.filter(
    (col("customer_age") >= 18)
    & (col("customer_age") <= 100)
    & (col("income") > 0)
)

In [0]:
display(silver_df.limit(10))

print(f"Silver rows: {silver_df.count()}")
print(f"Silver columns: {len(silver_df.columns)}")

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE_NAME)
)

In [0]:
silver_check_df = spark.table(SILVER_TABLE_NAME)

print(f"Rows saved: {silver_check_df.count()}")
print(f"Columns saved: {len(silver_check_df.columns)}")

display(silver_check_df.limit(10))